# 07 — Redundância do DNS autoritativo

**Perguntas:**
1. As empresas distribuem os seus servidores de nomes (NS) entre **várias organizações**, ou
   ficam concentradas em um único provedor?
2. As empresas que dependem de provedores **estrangeiros** mantêm pelo menos um provedor
   **doméstico** como alternativa?

**Escopo e definições (as mesmas dos notebooks 05 e 06):**
- Apenas IPv4.
- Provedor = grupo econômico dono do ASN (`dp_as_org`, depois do de-para).
- O país do provedor é o país do **grupo controlador** do ASN (`dp_as_country_code`), e não onde ficam os servidores.
- "Doméstico" = provedor do mesmo **país** da sede da empresa.

**Por que olhar organizações, e não só servidores?** Ter vários NS protege contra a queda
de *um servidor*. Mas se todos os NS estão no mesmo provedor, uma falha (ou bloqueio) desse
provedor derruba todos de uma vez. Por isso a redundância "de verdade" é medida pelo nº de
**organizações** diferentes.

## 1. Configuração e carga dos dados

In [28]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -----------------------------------------------------------------------------
# CONFIGURAÇÕES
# -----------------------------------------------------------------------------
# Aqui não existe a variável METODO dos notebooks 05 e 06: a redundância depende de
# QUANTOS provedores a empresa usa, então sempre considero todos os provedores dela.
# (no método "principal", por exemplo, toda empresa teria 1 provedor só)

# False -> cada empresa pesa 1 (contagem de empresas)
# True  -> cada empresa pesa a sua receita (revenue_ttm), ou seja, "quanto da economia"
PONDERAR_POR_RECEITA = False

# pastas de saída (tabelas em CSV e figuras em PNG)
PASTA_TAB = Path("../data/analysis")
PASTA_FIG = Path("../outputs/figures")
PASTA_TAB.mkdir(parents=True, exist_ok=True)
PASTA_FIG.mkdir(parents=True, exist_ok=True)

# sufixo colocado no nome dos arquivos, para as versões não se sobrescreverem
SUFIXO = "receita" if PONDERAR_POR_RECEITA else "empresas"

# texto usado no eixo dos gráficos, conforme a ponderação escolhida
UNIDADE = "revenue" if PONDERAR_POR_RECEITA else "companies"

print("Versão da análise:", SUFIXO)

Versão da análise: empresas


In [29]:
# -----------------------------------------------------------------------------
# PAÍSES E BLOCOS
# -----------------------------------------------------------------------------
# O CompaniesMarketCap informa o país da empresa por extenso ("Brazil"), enquanto o
# país do provedor vem em código ISO de 2 letras ("BR"). Para conseguir comparar os
# dois, converto o país da empresa para o código ISO.
PAIS_ISO2 = {
    "Argentina": "AR", "Australia": "AU", "Austria": "AT", "Bahamas": "BS", "Bahrain": "BH",
    "Belgium": "BE", "Bermuda": "BM", "Brazil": "BR", "British Virgin Islands": "VG",
    "Bulgaria": "BG", "Canada": "CA", "Cayman Islands": "KY", "Chile": "CL", "China": "CN",
    "Colombia": "CO", "Costa Rica": "CR", "Cyprus": "CY", "Czech Republic": "CZ",
    "Denmark": "DK", "Egypt": "EG", "Estonia": "EE", "Finland": "FI", "France": "FR",
    "Germany": "DE", "Gibraltar": "GI", "Greece": "GR", "Guernsey": "GG", "Hong Kong": "HK",
    "Hungary": "HU", "Iceland": "IS", "India": "IN", "Indonesia": "ID", "Ireland": "IE",
    "Isle of Man": "IM", "Israel": "IL", "Italy": "IT", "Japan": "JP", "Jersey": "JE",
    "Jordan": "JO", "Kazakhstan": "KZ", "Kuwait": "KW", "Latvia": "LV",
    "Liechtenstein": "LI", "Lithuania": "LT", "Luxembourg": "LU", "Macau": "MO",
    "Malaysia": "MY", "Malta": "MT", "Mexico": "MX", "Monaco": "MC", "Netherlands": "NL",
    "New Zealand": "NZ", "Norway": "NO", "Panama": "PA", "Peru": "PE", "Philippines": "PH",
    "Poland": "PL", "Portugal": "PT", "Qatar": "QA", "Romania": "RO", "Saudi Arabia": "SA",
    "Singapore": "SG", "South Africa": "ZA", "South Korea": "KR", "Spain": "ES",
    "Sweden": "SE", "Switzerland": "CH", "Taiwan": "TW", "Thailand": "TH", "Turkey": "TR",
    "United Arab Emirates": "AE", "United Kingdom": "GB", "United States": "US",
    "Uruguay": "UY", "Vietnam": "VN", "Russia": "RU", "Ethiopia": "ET", "Iran": "IR",
}

# União Europeia (27 países)
UE27 = {"AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU", "IE",
        "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE"}

# BRICS: os 5 membros originais + os membros plenos que entraram a partir de 2024/2025
# (deixei a Arábia Saudita de fora, porque a adesão formal dela é ambígua)
BRICS = {"BR", "RU", "IN", "CN", "ZA", "EG", "ET", "IR", "AE", "ID"}


def bloco_de(pais):
    """Recebe um código ISO de país e devolve o bloco: US, EU, BRICS ou Others."""
    if pais == "US":
        return "US"
    # alguns registros WHOIS trazem "EU" como país (registro regional europeu)
    if pais in UE27 or pais == "EU":
        return "EU"
    if pais in BRICS:
        return "BRICS"
    return "Others"


# ordem em que os blocos aparecem nas tabelas e gráficos ("Total" = todas as empresas)
ORDEM_BLOCOS = ["US", "EU", "BRICS", "Others", "Total"]
# PALETA PADRÃO DO TRABALHO: "tab10" do matplotlib. Todas as cores dos gráficos saem daqui.
#   0 azul | 1 laranja | 2 verde | 3 vermelho | 4 roxo | 5 marrom | 6 rosa | 7 cinza | 8 oliva | 9 ciano
PALETA = plt.get_cmap("tab10").colors

CORES_BLOCOS = {"US": PALETA[0], "EU": PALETA[1], "BRICS": PALETA[2],
                "Others": PALETA[7], "Total": PALETA[4]}

In [30]:
# -----------------------------------------------------------------------------
# ESTILO DOS GRÁFICOS E FUNÇÕES PARA SALVAR
# -----------------------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,      # tiro as bordas de cima e da direita
    "axes.spines.right": False,
    "axes.grid": True,             # grade horizontal leve, ajuda a ler os valores
    "axes.grid.axis": "y",
    "grid.color": "#e4e3df",
    "axes.axisbelow": True,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "legend.frameon": False,
})


def salvar_tabela(tabela, nome):
    """Salva a tabela em CSV, com o sufixo da versão da análise no nome."""
    tabela.to_csv(PASTA_TAB / f"{nome}_{SUFIXO}.csv")


def salvar_figura(fig, nome):
    """Salva a figura em PNG, com o sufixo da versão da análise no nome."""
    fig.savefig(PASTA_FIG / f"{nome}_{SUFIXO}.png", dpi=200, bbox_inches="tight")


def rotulos_com_n(n_por_bloco):
    """Cria os rótulos do eixo X no formato 'US\n(n=3,257)'."""
    return [f"{b}\n(n={n:,})" for b, n in n_por_bloco.items()]

In [31]:
# lê o dataset final (uma linha por empresa x NS x IP)
df = pd.read_parquet("../data/final/final_data.parquet")

# fico apenas com os endereços IPv4 (registros do tipo "A")
df = df[df["ip_type"] == "A"].copy()

# país e bloco da EMPRESA (onde fica a sede)
df["pais_empresa"] = df["company_country"].map(PAIS_ISO2)
df["bloco_empresa"] = df["pais_empresa"].apply(bloco_de)

# PROVEDOR = grupo econômico dono do ASN (já com o de-para aplicado)
df["provedor"] = df["dp_as_org"]
df["pais_provedor"] = df["dp_as_country_code"]
df["bloco_provedor"] = df["pais_provedor"].apply(bloco_de)

# conferência: nenhum país pode ter ficado sem código ISO
assert df["pais_empresa"].notna().all(), "há países sem código ISO no dicionário"

print(f"{len(df):,} linhas IPv4 | {df['rank'].nunique():,} empresas | {df['provedor'].nunique():,} provedores")

44,069 linhas IPv4 | 10,283 empresas | 1,083 provedores


## 2. Tabela base: uma linha por empresa

Primeiro resumo o dataset por par (empresa, provedor), como nos notebooks 05 e 06, e marco se
cada provedor é doméstico ou estrangeiro. Depois resumo de novo, agora com **uma linha por
empresa**, contando:
- `n_ns`: quantos servidores de nomes distintos a empresa tem;
- `n_provedores`: em quantas organizações diferentes esses servidores estão;
- `tem_domestico` / `tem_estrangeiro`: se ela usa pelo menos um provedor do próprio país / de outro país.

In [32]:
# 1) uma linha por (empresa, provedor), contando quantos NS a empresa tem naquele provedor
ep = (df.groupby(["rank", "provedor"], as_index=False)
        .agg(company_name=("company_name", "first"),
             revenue_ttm=("revenue_ttm", "first"),
             pais_empresa=("pais_empresa", "first"),
             bloco_empresa=("bloco_empresa", "first"),
             pais_provedor=("pais_provedor", "first"),
             bloco_provedor=("bloco_provedor", "first"),
             n_ns=("ns", "nunique")))

# 2) peso da empresa: 1 (contagem) ou a receita
ep["peso_empresa"] = ep["revenue_ttm"] if PONDERAR_POR_RECEITA else 1

# 3) o provedor é doméstico quando é do mesmo país da sede da empresa
#    (o código regional "EU" de alguns registros WHOIS nunca é igual a um país, então conta como estrangeiro)
ep["domestico"] = ep["pais_provedor"] == ep["pais_empresa"]
ep["estrangeiro"] = ~ep["domestico"]

# 4) para ter a coluna "Total" nas comparações, duplico a tabela trocando o bloco por "Total".
#    Assim, todo groupby por bloco já devolve US, EU, BRICS, Others e Total de uma vez.
base = pd.concat([ep, ep.assign(bloco_empresa="Total")], ignore_index=True)

# 5) resumo por empresa (dentro de cada bloco): é essa tabela que as análises usam
n_ns_empresa = df.groupby("rank")["ns"].nunique()   # nº de servidores de nomes distintos da empresa

por_empresa = base.groupby(["bloco_empresa", "rank"], as_index=False).agg(
    company_name=("company_name", "first"),
    n_provedores=("provedor", "nunique"),       # em quantas organizações os NS estão
    tem_domestico=("domestico", "any"),         # tem pelo menos um provedor do próprio país?
    tem_estrangeiro=("estrangeiro", "any"),     # tem pelo menos um provedor de outro país?
    peso_empresa=("peso_empresa", "first"),
)
por_empresa["n_ns"] = por_empresa["rank"].map(n_ns_empresa)

# nº de empresas de cada bloco (usado nos rótulos dos gráficos)
n_empresas = por_empresa.groupby("bloco_empresa")["rank"].nunique().reindex(ORDEM_BLOCOS)
display(n_empresas.to_frame("n_companies"))

# exemplo: as primeiras empresas da tabela
display(por_empresa[por_empresa["bloco_empresa"] != "Total"].head(8))

,n_companies
bloco_empresa,
US,3257
EU,1782
BRICS,1281
Others,3963
Total,10283


,bloco_empresa,rank,company_name,n_provedores,tem_domestico,tem_estrangeiro,peso_empresa,n_ns
0,BRICS,7,Sinopec,4,True,False,1,8
1,BRICS,9,PetroChina,2,True,False,1,2
2,BRICS,21,China State Construction Engineering,1,True,False,1,2
3,BRICS,35,JD.com,3,True,False,1,10
4,BRICS,43,Foxconn Industrial Internet,2,True,True,1,3
5,BRICS,46,China Railway Group,2,True,False,1,3
6,BRICS,49,Alibaba,1,True,False,1,4
7,BRICS,52,Ping An Insurance,1,True,False,1,3


## 3. Os servidores de nomes estão distribuídos entre várias organizações?

Praticamente todas as empresas têm 2 ou mais servidores de nomes (o percentual é impresso
logo abaixo), então o que interessa é **em quantas organizações** esses servidores estão.
Para cada bloco, separo as empresas em:
- **Single provider organization:** todos os NS no mesmo provedor (ponto único de falha);
- **2+ provider organizations:** NS distribuídos entre 2 ou mais provedores.

In [33]:
# contexto: quase todas as empresas têm 2+ servidores de nomes, por isso não entram no gráfico
print(f"Empresas com 2+ servidores de nomes: {100 * (por_empresa['n_ns'] >= 2).mean():.1f}%")

# classifico cada empresa pelo nº de organizações onde estão os seus NS
por_empresa["redundancia"] = "Single provider organization"
por_empresa.loc[por_empresa["n_provedores"] >= 2, "redundancia"] = "2+ provider organizations"

# tabela: % das empresas (ou da receita) de cada bloco em cada situação
ORDEM_REDUNDANCIA = ["Single provider organization", "2+ provider organizations"]
tab_redundancia = (por_empresa.pivot_table(index="bloco_empresa", columns="redundancia",
                                           values="peso_empresa", aggfunc="sum")
                              .reindex(index=ORDEM_BLOCOS, columns=ORDEM_REDUNDANCIA)
                              .fillna(0))
tab_redundancia = 100 * tab_redundancia.div(tab_redundancia.sum(axis=1), axis=0)   # % de cada linha

# coluna extra só para a tabela: nº médio de organizações por empresa
tab_redundancia["mean providers per company"] = por_empresa.groupby("bloco_empresa")["n_provedores"].mean()

display(tab_redundancia.round(1))
salvar_tabela(tab_redundancia, "red_provedores_blocos")

Empresas com 2+ servidores de nomes: 99.6%


redundancia,Single provider organization,2+ provider organizations,mean providers per company
bloco_empresa,,,
US,93.6,6.4,1.1
EU,76.8,23.2,1.3
BRICS,80.0,20.0,1.4
Others,86.8,13.2,1.2
Total,86.3,13.7,1.2


In [ ]:
# GRÁFICO 1: barras lado a lado (para cada bloco, uma barra por situação; as duas somam 100%)
cores_redundancia = [PALETA[3], PALETA[2]]   # vermelho (1 organização) e verde (2+ organizações)

fig, ax = plt.subplots(figsize=(8, 4.8))
tab_redundancia[ORDEM_REDUNDANCIA].plot.bar(ax=ax, width=0.75, edgecolor="white",
                                            color=cores_redundancia)

# escrevo o % em cima de cada barra
for barras in ax.containers:
    ax.bar_label(barras, fmt="%.0f%%", padding=2, fontsize=8)

ax.set_title("Are name servers spread across provider organizations?")
ax.grid(axis="x", visible=False)   # o pandas liga a grade vertical; deixo só a horizontal
ax.set_xlabel("")
ax.set_ylabel(f"Share of {UNIDADE} (%)")
ax.set_xticklabels(rotulos_com_n(n_empresas), rotation=0)
ax.set_ylim(0, 105)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")

salvar_figura(fig, "fig_red_provedores_blocos")
plt.show()

## 4. Quem depende de provedor estrangeiro mantém uma alternativa doméstica?

Para não olhar só um pedaço da amostra, classifico **todas** as empresas de cada bloco em
quatro situações, combinando a origem dos provedores com a redundância:

| Situação | Significado |
|---|---|
| **Domestic only** | usa só provedores do próprio país (não depende de estrangeiro) |
| **Foreign + domestic alternative** | usa provedor estrangeiro, mas mantém um do próprio país (é o *mixed* do notebook 06) |
| **Foreign only, 2+ providers** | só provedores estrangeiros, mas divide os NS entre 2 ou mais deles |
| **Foreign only, single provider** | depende de um único provedor, e ele é estrangeiro: o caso mais frágil |

As três últimas somadas são as empresas que dependem de algum provedor estrangeiro.

In [35]:
# classificação de TODAS as empresas
# (a ordem importa: cada linha sobrescreve a anterior quando a condição vale)
por_empresa["situacao"] = "Foreign only, single provider"
por_empresa.loc[por_empresa["n_provedores"] >= 2, "situacao"] = "Foreign only, 2+ providers"
por_empresa.loc[por_empresa["tem_domestico"] & por_empresa["tem_estrangeiro"], "situacao"] = "Foreign + domestic alternative"
por_empresa.loc[~por_empresa["tem_estrangeiro"], "situacao"] = "Domestic only"

# tabela: % das empresas (ou da receita) de cada bloco em cada situação
ORDEM_SITUACOES = ["Domestic only", "Foreign + domestic alternative",
                   "Foreign only, 2+ providers", "Foreign only, single provider"]
tab_alternativa = (por_empresa.pivot_table(index="bloco_empresa", columns="situacao",
                                           values="peso_empresa", aggfunc="sum")
                              .reindex(index=ORDEM_BLOCOS, columns=ORDEM_SITUACOES)
                              .fillna(0))
tab_alternativa = 100 * tab_alternativa.div(tab_alternativa.sum(axis=1), axis=0)   # % de cada linha

display(tab_alternativa.round(1))
salvar_tabela(tab_alternativa, "red_alternativa_domestica_blocos")

situacao,Domestic only,Foreign + domestic alternative,"Foreign only, 2+ providers","Foreign only, single provider"
bloco_empresa,,,,
US,80.0,0.6,1.4,18.0
EU,30.4,10.4,7.0,52.2
BRICS,41.1,3.7,2.5,52.7
Others,24.1,6.1,3.5,66.3
Total,45.0,4.8,3.3,46.9


In [ ]:
# GRÁFICO 2: barras lado a lado (para cada bloco, uma barra por situação; as quatro somam 100%)
# azul (doméstico, como no notebook 06), depois verde, laranja e vermelho (do melhor para o pior caso)
cores_situacoes = [PALETA[0], PALETA[2], PALETA[1], PALETA[3]]

fig, ax = plt.subplots(figsize=(8, 4.8))
tab_alternativa[ORDEM_SITUACOES].plot.bar(ax=ax, width=0.8, edgecolor="white",
                                          color=cores_situacoes)

# escrevo o % em cima de cada barra
for barras in ax.containers:
    ax.bar_label(barras, fmt="%.0f%%", padding=2, fontsize=8)

ax.set_title("Foreign dependence and domestic alternatives")
ax.grid(axis="x", visible=False)   # o pandas liga a grade vertical; deixo só a horizontal
ax.set_xlabel("")
ax.set_ylabel(f"Share of {UNIDADE} (%)")
ax.set_xticklabels(rotulos_com_n(n_empresas), rotation=0)
ax.set_ylim(0, 105)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")

salvar_figura(fig, "fig_red_alternativa_domestica_blocos")
plt.show()

## 5. Notas para a leitura

- A redundância é medida só no DNS **autoritativo** do domínio principal da empresa, em IPv4.
  Não capta redundância em subdomínios, na hospedagem ou em IPv6.
- Um provedor *anycast* (Cloudflare, Akamai etc.) tem servidores no mundo todo, então "1 provedor"
  não significa "1 servidor". Ainda assim, continua sendo **uma única organização**: um ponto
  único de falha do ponto de vista organizacional e jurisdicional.
- A falta de alternativa doméstica deve ser lida junto com a **oferta** de provedores no país:
  em países sem provedor local relevante, depender do estrangeiro não é exatamente uma escolha.
- Para gerar a versão ponderada por receita, basta mudar `PONDERAR_POR_RECEITA` na primeira
  célula de código e rodar tudo de novo. Os arquivos saem com o sufixo da versão no nome.